# LEC player rating — version 0.1 (early + late)

**How to use:** open this file in VS Code, pick your Python kernel (top right), then press **Run All**.
Change settings only in the first code cell (the CONFIG cell). Everything else is calculated from the Oracle's Elixir *match data* CSV.

**What it does**
1. Loads the LEC games you choose (all of 2026 by default, regular season only).
2. Rates every player in every game: an **early** rating (laning phase, up to minute 15), a **late** rating (after minute 15) and an **overall** rating.
3. Averages the game ratings per player, shrinks small samples toward 1.00 and rescales within each role.
4. Runs validation tests.
5. Exports CSV and JSON files for the website.

**1.00 = the average player of that role in the games you loaded.** Ratings are only ever compared within a role.

In [ ]:
# ================= CONFIG: change these, then Run All =================
FILE = "2026_LoL_esports_match_data_from_OraclesElixir.csv"   # must be in the same folder as this notebook
LEAGUE = "LEC"
SPLITS = None            # None = every split in the file, or e.g. ["Summer"] or ["Spring", "Summer"]
INCLUDE_PLAYOFFS = False
ONLY_COMPLETE = True     # drop games whose data is marked partial
MIN_GAMES = 10           # players with fewer games are rated but not ranked
K = 3                    # shrinkage: higher pulls small-sample players harder toward 1.00
GAME_SD = 0.25           # spread of single-game ratings
PLAYER_SD = 0.15         # spread of final player ratings within a role
OUT_DIR = "lec_rating_output"

# --- EARLY score: weights of the z-scores (they add up to 1) ---
EARLY_WEIGHTS = {
    "golddiffat10": .15, "golddiffat15": .25,            # gold lead vs lane opponent: 40%
    "killsat15": .22, "assistsat15": .08, "deathsat15": .15,   # fighting before minute 15
    "xpdiffat15": .10,                                   # level lead
    "fb_score": .05,                                     # first blood involvement
}

# --- LATE score: groups of stats (each group = plain average of its z-scores) ---
LATE_GROUPS = {
    "fights":   ["late_kp", "late_deathshare", "damageshare", "dpm"],
    "swing":    ["gold_prog"],                           # gold-diff change per minute after minute 15
    "vision":   ["vspm", "wpm", "wcpm", "cwpm"],
    "pressure": ["towerdmg_pm"],
}
# role-specific weights of the groups (each row adds up to 1)
LATE_WEIGHTS = {
    "top": {"fights": .40, "swing": .25, "vision": .05, "pressure": .30},
    "jng": {"fights": .40, "swing": .20, "vision": .15, "pressure": .25},
    "mid": {"fights": .50, "swing": .25, "vision": .05, "pressure": .20},
    "bot": {"fights": .50, "swing": .30, "vision": .05, "pressure": .15},
    "sup": {"fights": .35, "swing": .15, "vision": .40, "pressure": .10},
}
# overall = share * early + (1 - share) * late
EARLY_SHARE = {"top": .40, "jng": .35, "mid": .35, "bot": .35, "sup": .25}

# +1 = higher is better (default), -1 = lower is better
SIGNS = {"deathsat15": -1, "deathshare": -1, "late_deathshare": -1}

In [ ]:
import os, json
from datetime import datetime
import numpy as np
import pandas as pd

df = pd.read_csv(FILE, low_memory=False)
print(f"{LEAGUE} splits in this file:", sorted(df.loc[df["league"] == LEAGUE, "split"].dropna().unique()))

mask = (df["league"] == LEAGUE)
if SPLITS is not None:
    mask &= df["split"].isin(SPLITS)
if not INCLUDE_PLAYOFFS:
    mask &= (df["playoffs"] == 0)
p = df[mask & (df["position"] != "team")].copy()

if ONLY_COMPLETE and "datacompleteness" in p.columns:
    n_before = p["gameid"].nunique()
    p = p[p["datacompleteness"] == "complete"].copy()
    print("games dropped for incomplete data:", n_before - p["gameid"].nunique())

# keep only games with exactly 10 player rows
p = p[p.groupby("gameid")["gameid"].transform("size") == 10].copy()

p["date"] = pd.to_datetime(p["date"])
p = p.sort_values(["date", "gameid"]).reset_index(drop=True)
print("games:", p["gameid"].nunique(), "| players:", p["playername"].nunique(),
      "|", p["date"].min().date(), "to", p["date"].max().date())
print(p.groupby("split")["gameid"].nunique().to_string())

In [ ]:
# ---------- raw per-game stats ----------
num_cols = ["gamelength", "result", "kills", "deaths", "assists", "teamkills", "teamdeaths",
            "firstbloodkill", "firstbloodassist", "firstbloodvictim", "damageshare", "dpm",
            "earned gpm", "earnedgoldshare", "cspm", "vspm", "wpm", "wcpm",
            "controlwardsbought", "damagetotowers"]
for t_ in (10, 15, 20, 25):
    num_cols += [f"golddiffat{t_}", f"xpdiffat{t_}", f"csdiffat{t_}",
                 f"killsat{t_}", f"assistsat{t_}", f"deathsat{t_}"]
for c in num_cols:
    if c in p.columns:
        p[c] = pd.to_numeric(p[c], errors="coerce")
    else:
        print("column missing in file:", c)

p["minutes"] = p["gamelength"] / 60
tk = p["teamkills"].replace(0, np.nan)
td = p["teamdeaths"].replace(0, np.nan)
p["kp"] = (p["kills"] + p["assists"]) / tk
p["killshare"] = p["kills"] / tk
p["deathshare"] = p["deaths"] / td
p["cwpm"] = p["controlwardsbought"] / p["minutes"]
p["towerdmg_pm"] = p["damagetotowers"] / p["minutes"]
p["fb_score"] = (p["firstbloodkill"].fillna(0) + 0.5 * p["firstbloodassist"].fillna(0)
                 - p["firstbloodvictim"].fillna(0))

# ---------- stats after minute 15 = full-game totals minus the @15 values ----------
p["team_k15"] = p.groupby(["gameid", "teamname"])["killsat15"].transform("sum")
p["team_d15"] = p.groupby(["gameid", "teamname"])["deathsat15"].transform("sum")
late_tk = p["teamkills"] - p["team_k15"]
late_td = p["teamdeaths"] - p["team_d15"]
ok = (p["minutes"] > 18) & (late_tk > 0) & (late_td > 0)      # needs a real late game
late_ka = (p["kills"] - p["killsat15"]) + (p["assists"] - p["assistsat15"])
p["late_kp"] = (late_ka / late_tk).where(ok)
p["late_deathshare"] = ((p["deaths"] - p["deathsat15"]) / late_td).where(ok)

# gold-diff change per minute after minute 15 (15->25 if the game reached 25, otherwise 15->20)
d15, d20, d25 = p["golddiffat15"], p["golddiffat20"], p["golddiffat25"]
p["gold_prog"] = ((d25 - d15) / 10).where(d25.notna(), (d20 - d15) / 5)

# opponent team, for display
teams_in_game = p.groupby("gameid")["teamname"].agg(lambda s: sorted(set(s)))
p["opponent"] = [next((x for x in teams_in_game[g] if x != t_), None)
                 for g, t_ in zip(p["gameid"], p["teamname"])]
print("raw stats ready")

In [ ]:
# ---------- z-scores: each stat compared only with the SAME ROLE, clipped to +-3 ----------
def zscore(col):
    g = p.groupby("position")[col]
    z = (p[col] - g.transform("mean")) / g.transform("std")
    return (SIGNS.get(col, 1) * z).clip(-3, 3)

all_stats = list(EARLY_WEIGHTS) + [s for grp in LATE_GROUPS.values() for s in grp]
for s in dict.fromkeys(all_stats):
    if s in p.columns and p[s].notna().mean() >= 0.5:
        p["z_" + s] = zscore(s)
    else:
        print("skipping stat (missing or mostly empty):", s)

def blend(frame, weights):
    """Weighted average across columns, ignoring missing values (weights re-scaled per row)."""
    Z = frame.to_numpy(dtype=float)
    W = np.asarray(weights, dtype=float)
    present = ~np.isnan(Z)
    num = np.where(present, Z, 0.0) @ W
    den = present.astype(float) @ W
    out = np.full(len(Z), np.nan)
    np.divide(num, den, out=out, where=den > 0)
    return pd.Series(out, index=frame.index)

# ---------- EARLY score ----------
ecols = [s for s in EARLY_WEIGHTS if "z_" + s in p.columns]
p["early"] = blend(p[["z_" + s for s in ecols]], [EARLY_WEIGHTS[s] for s in ecols])

# ---------- LATE score ----------
for name, stats in LATE_GROUPS.items():
    cols = ["z_" + s for s in stats if "z_" + s in p.columns]
    p["lg_" + name] = blend(p[cols], [1.0] * len(cols)) if cols else np.nan

p["late"] = np.nan
for role, w in LATE_WEIGHTS.items():
    m = p["position"] == role
    groups = [g for g in w if "lg_" + g in p.columns]
    p.loc[m, "late"] = blend(p.loc[m, ["lg_" + g for g in groups]], [w[g] for g in groups])

# ---------- OVERALL = blend of early and late (early only if a game has no late score) ----------
share = p["position"].map(EARLY_SHARE)
both = p["early"].notna() & p["late"].notna()
p["overall"] = np.where(both, share * p["early"] + (1 - share) * p["late"], p["early"].fillna(p["late"]))

# ---------- game ratings: 1.00 = average game of that role ----------
def to_rating(col):
    g = p.groupby("position")[col]
    return 1 + GAME_SD * (p[col] - g.transform("mean")) / g.transform("std")

for c in ["early", "late", "overall"]:
    p["r_" + c] = to_rating(c)

print("player-games without a late score:", int(p["late"].isna().sum()), "| without an early score:", int(p["early"].isna().sum()))

In [ ]:
# ---------- player table: average, shrink, rescale within role ----------
z_cols = ([f"z_{s}" for s in EARLY_WEIGHTS if f"z_{s}" in p.columns]
          + [c for c in p.columns if c.startswith("lg_")])
g = p.groupby(["playername", "position"])
t = g.agg(team=("teamname", "last"),
          teams=("teamname", lambda s: ", ".join(dict.fromkeys(s))),
          games=("gameid", "nunique"),
          win_rate=("result", "mean"),
          early=("r_early", "mean"), late=("r_late", "mean"), overall=("r_overall", "mean"),
          early_n=("r_early", "count"), late_n=("r_late", "count"), overall_n=("r_overall", "count"),
          consistency=("r_overall", "std")).reset_index()
t = t.merge(g[z_cols].mean().round(2).reset_index(), on=["playername", "position"])

t["qualified"] = t["games"] >= MIN_GAMES
for c in ["early", "late", "overall"]:
    n = t[c + "_n"]
    shrunk = 1 + n / (n + K) * (t[c] - 1)
    ref = shrunk[t["qualified"]].groupby(t.loc[t["qualified"], "position"])
    mu = t["position"].map(ref.mean())
    sd = t["position"].map(ref.std())
    t[c + "_final"] = 1 + PLAYER_SD * (shrunk - mu) / sd

t["rank_in_role"] = (t[t["qualified"]].groupby("position")["overall_final"]
                     .rank(ascending=False, method="min")).astype("Int64")
t = t.sort_values(["position", "rank_in_role"]).reset_index(drop=True)

show_cols = ["rank_in_role", "playername", "team", "games", "early_final", "late_final",
             "overall_final", "consistency", "win_rate"]
for pos in ["top", "jng", "mid", "bot", "sup"]:
    print(pos.upper())
    print(t[(t["position"] == pos) & t["qualified"]][show_cols].round(2).to_string(index=False))
    print()

In [ ]:
# ================= VALIDATION =================
validation = {}

# 1) Winners should rate higher overall, but the EARLY rating should be more even between winners and losers
gap = p.groupby("result")[["r_early", "r_late", "r_overall"]].mean().round(3)
print("Average game rating by result (1 = win):")
print(gap.to_string(), "\n")
validation["avg_rating_loss"] = {c: float(gap.loc[0, c]) for c in gap.columns}
validation["avg_rating_win"] = {c: float(gap.loc[1, c]) for c in gap.columns}

# 2) How much does each rating just copy the gold difference at 15 minutes?
corr = {c: round(float(p[c].corr(p["golddiffat15"])), 2) for c in ["r_early", "r_late", "r_overall"]}
print("Correlation with gold diff @15:", corr, "\n")
validation["corr_with_golddiff15"] = corr

# 3) Split-half reliability: rate each player on two halves of their games. If the rating measures
#    something real, the two halves should rank players similarly (1.0 = identical).
def split_half(col, how, min_games=4):
    q = p.dropna(subset=[col]).copy()
    key = ["playername", "position"]
    q["_i"] = q.groupby(key).cumcount()
    q["_n"] = q.groupby(key)[col].transform("size")
    q["_half"] = (q["_i"] % 2) if how == "odd/even" else (q["_i"] >= q["_n"] / 2).astype(int)
    h = q.groupby(key + ["_half"])[col].agg(["mean", "size"]).reset_index()
    a = h[h["_half"] == 0].set_index(key)
    b = h[h["_half"] == 1].set_index(key)
    j = a.join(b, lsuffix="_a", rsuffix="_b", how="inner")
    j = j[(j["size_a"] >= min_games) & (j["size_b"] >= min_games)]
    r = j["mean_a"].rank().corr(j["mean_b"].rank())            # Spearman correlation
    return round(float(r), 2), round(float(2 * r / (1 + r)), 2), int(len(j))

print("Split-half reliability (rank correlation | corrected to full length | players)")
validation["split_half"] = {}
for col in ["r_early", "r_late", "r_overall"]:
    for how in ["odd/even", "first/second half"]:
        r, r_full, n_pl = split_half(col, how)
        print(f"  {col:10s} {how:18s} {r:5.2f} | {r_full:5.2f} | {n_pl} players")
        validation["split_half"][f"{col} {how}"] = {"r": r, "r_full": r_full, "players": n_pl}
print()

# 4) Out-of-sample test: predict each game from the players' average ratings in all OTHER games
key = ["playername", "position"]
tot = p.groupby(key)["r_overall"].transform("sum")
cnt = p.groupby(key)["r_overall"].transform("count")
p["loo"] = ((tot - p["r_overall"]) / (cnt - 1)).where(cnt > 1)

ts = p.groupby(["gameid", "teamname"]).agg(strength=("loo", "mean"), result=("result", "first")).reset_index()
wins = ts.groupby("teamname")["result"].transform("sum")
n_g = ts.groupby("teamname")["result"].transform("size")
ts["team_wr_loo"] = ((wins - ts["result"]) / (n_g - 1)).where(n_g > 1)
pairs = ts.merge(ts, on="gameid", suffixes=("", "_o"))
pairs = pairs[pairs["teamname"] != pairs["teamname_o"]]

def accuracy(col):
    x = pairs.dropna(subset=[col, col + "_o"])
    x = x[x[col] != x[col + "_o"]]
    return round(float(((x[col] > x[col + "_o"]) == (x["result"] == 1)).mean()), 3), int(len(x) // 2)

acc_r, n_r = accuracy("strength")
acc_w, n_w = accuracy("team_wr_loo")
print(f"Predicting the winner from player ratings (other games only): {acc_r:.1%} correct over {n_r} games")
print(f"Baseline, predicting from each team's win rate in other games: {acc_w:.1%} over {n_w} games")
validation["predict_accuracy_player_ratings"] = {"accuracy": acc_r, "games": n_r}
validation["predict_accuracy_team_winrate"] = {"accuracy": acc_w, "games": n_w}

In [ ]:
# ---------- look at one player, game by game ----------
def show_player(name):
    x = p[p["playername"] == name].copy()
    x["date"] = x["date"].dt.strftime("%Y-%m-%d")
    x["KDA"] = (x["kills"].fillna(0).astype(int).astype(str) + "/" + x["deaths"].fillna(0).astype(int).astype(str)
                + "/" + x["assists"].fillna(0).astype(int).astype(str))
    x["min"] = x["minutes"].round(1)
    cols = ["date", "opponent", "champion", "result", "min", "KDA", "golddiffat15", "r_early", "r_late", "r_overall"]
    print(x[cols].round(2).to_string(index=False))
    print()
    print(t[t["playername"] == name][["position", "team", "games", "early_final", "late_final", "overall_final", "rank_in_role"]]
          .round(2).to_string(index=False))

show_player(t.loc[t["qualified"], "playername"].iloc[0])     # change to any player name, e.g. show_player("Lyncas")

In [ ]:
# ================= EXPORT =================
os.makedirs(OUT_DIR, exist_ok=True)

def records(frame):
    frame = frame.astype(object).where(frame.notna(), None)
    return frame.to_dict(orient="records")

# --- games ---
game_cols = ["gameid", "date", "split", "playoffs", "playername", "position", "teamname", "opponent", "champion",
             "result", "minutes", "kills", "deaths", "assists", "golddiffat15", "xpdiffat15", "csdiffat15",
             "r_early", "r_late", "r_overall"]
games_out = p[game_cols].copy()
games_out["date"] = games_out["date"].dt.strftime("%Y-%m-%d %H:%M")
games_out["minutes"] = games_out["minutes"].round(1)
games_out[["r_early", "r_late", "r_overall"]] = games_out[["r_early", "r_late", "r_overall"]].round(3)
games_out.to_csv(os.path.join(OUT_DIR, "lec_games.csv"), index=False)

# --- players ---
player_cols = (["playername", "position", "team", "teams", "games", "qualified", "rank_in_role",
                "early_final", "late_final", "overall_final", "consistency", "win_rate"] + z_cols)
players_out = t[player_cols].copy()
for c in ["early_final", "late_final", "overall_final", "consistency", "win_rate"]:
    players_out[c] = players_out[c].round(3)
players_out.to_csv(os.path.join(OUT_DIR, "lec_players.csv"), index=False)

# --- one JSON file for the website ---
gl = games_out.copy()
gl["kda"] = (gl["kills"].fillna(0).astype(int).astype(str) + "/" + gl["deaths"].fillna(0).astype(int).astype(str)
             + "/" + gl["assists"].fillna(0).astype(int).astype(str))
gl = gl.rename(columns={"teamname": "team"})[["playername", "position", "date", "split", "opponent", "champion",
                                                "result", "minutes", "kda", "golddiffat15",
                                                "r_early", "r_late", "r_overall"]]
by_player = {k: records(v.drop(columns=["playername", "position"]))
             for k, v in gl.groupby(["playername", "position"])}
players_json = records(players_out)
for rec in players_json:
    rec["game_list"] = by_player.get((rec["playername"], rec["position"]), [])

payload = {
    "version": "0.1",
    "generated": datetime.now().isoformat(timespec="seconds"),
    "sample": {"league": LEAGUE, "splits": SPLITS if SPLITS else "all", "playoffs_included": INCLUDE_PLAYOFFS,
               "games": int(p["gameid"].nunique()), "players": int(p["playername"].nunique()),
               "date_from": str(p["date"].min().date()), "date_to": str(p["date"].max().date())},
    "formula": {"early_weights": EARLY_WEIGHTS, "late_groups": LATE_GROUPS, "late_weights": LATE_WEIGHTS,
                "early_share": EARLY_SHARE, "shrinkage_K": K, "game_sd": GAME_SD, "player_sd": PLAYER_SD,
                "min_games": MIN_GAMES},
    "validation": validation,
    "players": players_json,
}
with open(os.path.join(OUT_DIR, "lec_ratings.json"), "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, default=str)

for name in sorted(os.listdir(OUT_DIR)):
    print(f"{name:20s} {os.path.getsize(os.path.join(OUT_DIR, name)) / 1024:8.0f} KB")
print("Saved in:", os.path.abspath(OUT_DIR))